# Практическая работа №3: Обработка выборочных данных. Нахождение интервальных оценок параметров распределения. Проверка статистической гипотезы о нормальном распределении

Выполнили студенты гр. 2383 Борисов Иван Вадимович и Сыздыков Нургалым Конакбаевич. Вариант №25

## Цель работы

Получение практических навыков вычисления интервальных статистических оценок параметров распределения выборочных данных и проверки «справедливости» статистических гипотез. 


## Краткое изложение основных теоретических понятий

### Доверительный интервал для математического ожидания
Для неизвестного среднеквадратичного отклонения доверительный интервал вычисляется на основе распределения Стьюдента:
$$\left( \bar{x}_в - \frac{t_{\gamma} S}{\sqrt{n}}; \bar{x}_в + \frac{t_{\gamma} S}{\sqrt{n}} \right)$$
где $t_{\gamma}$ — квантиль распределения Стьюдента с $n-1$ степенями свободы, а точность оценки $\delta = \frac{t_{\gamma} S}{\sqrt{n}}$.

### Доверительный интервал для среднеквадратичного отклонения
Доверительный интервал для $\sigma$ строится из двойного неравенства:
$$S(1-q) < \sigma < S(1+q)$$
Само значение $q$ определяется при заданных $\gamma$ и $n$.  из интегрального уравнения для распределения $\chi$, которое эквивалентно:
$$\mathbb{P}\left(\frac{\sqrt{n-1}}{1+q} < \chi < \frac{\sqrt{n-1}}{1-q}\right) = \gamma$$

**Доверительный интервал** — это интервал $(\theta^* - \delta; \theta^* + \delta)$, который с заданной вероятностью $\gamma$ покрывает истинное значение оцениваемого параметра $\theta$:
$$\mathbb{P}(\theta^* - \delta < \theta < \theta^* + \delta) = \gamma$$
где $\delta$ — точность оценки.

**Статистическая гипотеза** — это предположение о виде неизвестного распределения или о параметрах известных распределений. 
* **Нулевая гипотеза ($H_0$)** — основное проверяемое предположение (например, данные распределены нормально).
* **Конкурирующая (альтернативная) гипотеза ($H_1$)** — предположение, противоречащее нулевой гипотезе.

**Критерий согласия Пирсона ($\chi^2$)** — статистический критерий, позволяющий оценить уровень значимости различий между эмпирическим (наблюдаемым) и теоретическим распределениями.

## Постановка задачи с кратким описанием порядка выполнения работы

Для заданной надежности определить (на основании выборочных данных и результатов выполнения практической работы №2) границы доверительных интервалов для математического ожидания и среднеквадратичного отклонения случайной величины. Проверить гипотезу о нормальном распределении исследуемой случайной величины с помощью критерия Пирсона $\chi^2$. Дать содержательную интерпретацию полученным результатам. 

### Порядок выполнения работы

1. Вычислить точность и доверительный интервал для математического ожидания при неизвестном среднеквадратичном отклонении при заданном объёме выборки для доверительной точности $\gamma \in \{0.95,0.99\}$. Сделать выводы.
2. Для вычисления границ доверительного интервала для среднеквадратичного отклонения определить значение $q$ при заданных $\gamma$ и $n$. Построить доверительные интервалы, сделать выводы.
3. Проверить гипотезу о нормальности заданного распределения с помощью критерия $\chi^2$ (Пирсона). Для этого необходимо найти теоретические частоты и вычислить наблюдаемое значение критерия. Для удобства вычисления необходимо заполнить табл. 1.
4. Доказать, что $$\chi^2_{набл} = \sum_{i=1}^{k} \frac{n_i^2}{n_i'} - n$$
   Проконтролировать корректность вычисления $\chi^2_{набл}$.
5. По заданному уровню значимости $\alpha = 0.05$ и числу степеней свободы $df$ найти критическую точку $\chi^2_{крит}$ и сравнить с наблюдаемым значением. Сделать выводы.

### 1. Вычислить точность и доверительный интервал для математического ожидания при неизвестном среднеквадратичном отклонении при заданном объёме выборки для доверительной точности $\gamma \in \{0.95,0.99\}$. Сделать выводы.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import brentq
from scipy.stats import t, chi, norm, chi2
from IPython.display import display, Markdown


pd.options.display.float_format = '{:,.4f}'.format

In [2]:
df: pd.DataFrame = pd.read_csv('Syzdykov_Borisov_data_116.csv')
n: int = len(df)

In [3]:
df

,nu,E
0,396,90.1000
1,525,145.3000
2,437,129.4000
3,411,115.2000
4,510,124.1000
...,...,...
111,463,144.9000
112,487,146.0000
113,528,163.4000
114,331,74.1000


In [4]:
def get_interval_stats(series: pd.Series) -> tuple[np.ndarray, np.ndarray, float, float]:
    n: int = len(series)
    k: int = int(np.ceil(1 + 3.31 * np.log10(n)))
    
    x_min: float = series.min()
    x_max: float = series.max()
    h: float = (x_max - x_min) / k
    
    bins: np.ndarray = np.linspace(x_min, x_max, k + 1)
    counts, edges = np.histogram(series, bins=bins)
    midpoints: np.ndarray = (edges[:-1] + edges[1:]) / 2
    
    return midpoints, counts, h, x_min
    

def compute_grouped_statistics(series: pd.Series):
    midpoints, counts, h, x_min = get_interval_stats(series)
    n = len(series)
    k = len(midpoints)
    
    edges = np.linspace(x_min, x_min + k * h, k + 1)
    x_mean = np.sum(midpoints * counts) / n
    D_v = np.sum(counts * (midpoints - x_mean)**2) / n
    S = np.sqrt(D_v * n / (n - 1))
    
    return {
        'n': n,
        'k': k,
        'h': h,
        'midpoints': midpoints,
        'counts': counts,
        'edges': edges,
        'mean': x_mean,
        'S': S
    }


stats_nu = compute_grouped_statistics(df['nu'])
stats_E = compute_grouped_statistics(df['E'])

print("Признак 'nu':")
print(f"Выборочное среднее x̄ = {stats_nu['mean']:.4f}, Выборочное СКО S = {stats_nu['S']:.4f}\n")

print("Признак 'E':")
print(f"Выборочное среднее x̄ = {stats_E['mean']:.4f}, Выборочное СКО S = {stats_E['S']:.4f}")

Признак 'nu':
Выборочное среднее x̄ = 455.8276, Выборочное СКО S = 57.0741

Признак 'E':
Выборочное среднее x̄ = 128.1636, Выборочное СКО S = 22.0210


Вычислим точность и доверительный интервал для $\gamma \in \{0.95, 0.99\}$.

In [5]:
def expected_value_interval(stats_dict, gammas=[0.95, 0.99]):
    n = stats_dict['n']
    x_mean = stats_dict['mean']
    S = stats_dict['S']
    df_t = n - 1
    
    for gamma in gammas:
        # Для двустороннего интервала
        t_gamma = t.ppf((1 + gamma) / 2, df_t)
        delta = t_gamma * S / np.sqrt(n)
        
        left_bound = x_mean - delta
        right_bound = x_mean + delta
        
        print(f"γ = {gamma}:")
        print(f"Δ = {delta:.4f}")
        print(f"Доверительный интервал: ({left_bound:.4f}; {right_bound:.4f})\n")

print("--- Признак 'nu' ---")
expected_value_interval(stats_nu)

print("--- Признак 'E' ---")
expected_value_interval(stats_E)

--- Признак 'nu' ---
γ = 0.95:
Δ = 10.4967
Доверительный интервал: (445.3309; 466.3243)

γ = 0.99:
Δ = 13.8800
Доверительный интервал: (441.9476; 469.7075)

--- Признак 'E' ---
γ = 0.95:
Δ = 4.0500
Доверительный интервал: (124.1136; 132.2135)

γ = 0.99:
Δ = 5.3553
Доверительный интервал: (122.8082; 133.5189)



**Вывод:**
Из полученных результатов видно, что с увеличением доверительной точности $\gamma$ (с $0.95$ до $0.99$) ширина доверительного интервала увеличивается, радиус интервала $\delta$ также возрастает. Это согласуется с теоретическими основами: для гарантии более высокой вероятности накрытия истинного параметра требуется более широкий интервал.

### 2. Для вычисления границ доверительного интервала для среднеквадратичного отклонения определить значение $q$ при заданных $\gamma$ и $n$. Построить доверительные интервалы, сделать выводы.

In [6]:
def find_q_value(n, gamma):
    df_chi = n - 1
    sqrt_n1 = np.sqrt(df_chi)
    def equation(q):
        if q <= 0 or q >= 1:
            return 1e9  # Выход за ОДЗ
        left_chi = sqrt_n1 / (1 + q)
        right_chi = sqrt_n1 / (1 - q)
        return chi.cdf(right_chi, df_chi) - chi.cdf(left_chi, df_chi) - gamma
    
    return brentq(equation, 0.001, 0.999)


def variance_confidence_interval(stats_dict, gammas=[0.95, 0.99]):
    n = stats_dict['n']
    S = stats_dict['S']
    
    for gamma in gammas:
        q = find_q_value(n, gamma)
        left = S * (1 - q)
        right = S * (1 + q)
        
        print(f"γ = {gamma}:")
        print(f"q = {q:.4f}")
        print(f"Доверительный интервал СКО: ({left:.4f}; {right:.4f})\n")

In [7]:
print("--- Признак 'nu' ---")
variance_confidence_interval(stats_nu)

print("--- Признак 'E' ---")
variance_confidence_interval(stats_E)

--- Признак 'nu' ---
γ = 0.95:
q = 0.1320
Доверительный интервал СКО: (49.5416; 64.6065)

γ = 0.99:
q = 0.1810
Доверительный интервал СКО: (46.7418; 67.4063)

--- Признак 'E' ---
γ = 0.95:
q = 0.1320
Доверительный интервал СКО: (19.1147; 24.9273)

γ = 0.99:
q = 0.1810
Доверительный интервал СКО: (18.0345; 26.0075)



**Вывод:** Доверительный интервал для среднеквадратичного отклонения также расширяется при росте $\gamma$. Вычисленное значение параметра $q$ увеличивается при переходе от $\gamma = 0.95$ к $\gamma = 0.99$.

### 3. Проверить гипотезу о нормальности заданного распределения с помощью критерия $\chi^2$ (Пирсона). Для этого необходимо найти теоретические частоты и вычислить наблюдаемое значение критерия. Для удобства вычисления необходимо заполнить табл. 1.

In [8]:
def perform_pearson_test(stats_dict):
    n_obs = stats_dict['n']
    mean_val = stats_dict['mean']
    std_val = stats_dict['S']
    counts = stats_dict['counts']
    edges = stats_dict['edges']
    
    norm_edges = np.array(edges, dtype=float)
    norm_edges[0] = -np.inf
    norm_edges[-1] = np.inf
    
    z_scores = (norm_edges - mean_val) / std_val
    p_teor = np.diff(norm.cdf(z_scores))
    n_expected = n_obs * p_teor
    
    grp_obs, grp_exp, grp_p, grp_bounds = [], [], [], []
    curr_obs, curr_exp, curr_p = 0, 0, 0
    left_b = edges[0]
    
    for i in range(len(counts)):
        curr_obs += counts[i]
        curr_exp += n_expected[i]
        curr_p += p_teor[i]
        
        if curr_exp >= 5 or i == len(counts) - 1:
            grp_obs.append(curr_obs)
            grp_exp.append(curr_exp)
            grp_p.append(curr_p)
            grp_bounds.append((left_b, edges[i+1]))
            
            curr_obs, curr_exp, curr_p = 0, 0, 0
            if i + 1 < len(edges):
                left_b = edges[i+1]
                
    while len(grp_exp) > 1 and grp_exp[-1] < 5:
        last_obs = grp_obs.pop()
        grp_obs[-1] += last_obs
        
        last_exp = grp_exp.pop()
        grp_exp[-1] += last_exp
        
        last_p = grp_p.pop()
        grp_p[-1] += last_p
        
        last_bound = grp_bounds.pop()
        grp_bounds[-1] = (grp_bounds[-1][0], last_bound[1])

    grp_obs = np.array(grp_obs)
    grp_exp = np.array(grp_exp)
    grp_p = np.array(grp_p)
    
    interval_labels = []
    for idx, (l, r) in enumerate(grp_bounds):
        str_l = f"{l:.2f}"
        str_r = f"{r:.2f}"
        interval_labels.append(f"[{str_l}; {str_r})")
        
    diff_sq = (grp_obs - grp_exp)**2
    frac_diff = diff_sq / grp_exp
    obs_sq = grp_obs**2
    frac_obs = obs_sq / grp_exp
    
    df_result = pd.DataFrame({
        'i': list(range(1, len(grp_obs) + 1)) + ['Σ'],
        '[x_i, x_{i+1})': interval_labels + ['-'],
        'n_i': list(grp_obs) + [grp_obs.sum()],
        'p_i': [round(x, 4) for x in grp_p] + [round(grp_p.sum(), 4)],
        "n'_i": [round(x, 4) for x in grp_exp] + [round(grp_exp.sum(), 4)],
        "(n_i - n'_i)^2": [round(x, 4) for x in diff_sq] + ['-'],
        "(n_i - n'_i)^2 / n'_i": [round(x, 4) for x in frac_diff] + [round(frac_diff.sum(), 4)],
        'n_i^2': list(obs_sq) + ['-'],
        "n_i^2 / n'_i": [round(x, 4) for x in frac_obs] + [round(frac_obs.sum(), 4)]
    })
    
    display(df_result)
    
    chi2_value = frac_diff.sum()
    alt_chi2_value = frac_obs.sum() - n_obs
    degrees_of_freedom_k = len(grp_obs)
    
    print(f"χ²_набл = {chi2_value:.4f}")
    print(f"χ²_набл (формула) = {alt_chi2_value:.4f}")
    
    return chi2_value, degrees_of_freedom_k, alt_chi2_value


print(">>> Таблица для признака 'nu':")
chi2_obs_nu, k_nu, alt_chi2_nu = perform_pearson_test(stats_nu)

print(">>> Таблица для признака 'E':")
chi2_obs_E, k_E, alt_chi2_E = perform_pearson_test(stats_E)

>>> Таблица для признака 'nu':


,i,"[x_i, x_{i+1})",n_i,p_i,n'_i,(n_i - n'_i)^2,(n_i - n'_i)^2 / n'_i,n_i^2,n_i^2 / n'_i
0,1,[320.00; 395.75),16,0.1463,16.9657,0.9327,0.0550,256,15.0892
1,2,[395.75; 433.62),22,0.2024,23.4757,2.1777,0.0928,484,20.6171
2,3,[433.62; 471.50),33,0.2596,30.1083,8.3619,0.2777,1089,36.1694
3,4,[471.50; 509.38),26,0.2177,25.2583,0.5501,0.0218,676,26.7635
4,5,[509.38; 547.25),14,0.1195,13.8586,0.0200,0.0014,196,14.1428
5,6,[547.25; 623.00),5,0.0546,6.3333,1.7777,0.2807,25,3.9474
6,Σ,-,116,1.0000,116.0000,-,0.7294,-,116.7294


χ²_набл = 0.7294
χ²_набл (формула) = 0.7294
>>> Таблица для признака 'E':


,i,"[x_i, x_{i+1})",n_i,p_i,n'_i,(n_i - n'_i)^2,(n_i - n'_i)^2 / n'_i,n_i^2,n_i^2 / n'_i
0,1,[71.10; 102.25),16,0.1196,13.8788,4.4997,0.3242,256,18.4455
1,2,[102.25; 117.82),15,0.1997,23.1671,66.7015,2.8791,225,9.7120
2,3,[117.82; 133.40),34,0.2746,31.8557,4.5981,0.1443,1156,36.2886
3,4,[133.40; 148.97),34,0.2337,27.1103,47.4673,1.7509,1156,42.6405
4,5,[148.97; 164.55),14,0.1231,14.2772,0.0769,0.0054,196,13.7281
5,6,[164.55; 195.70),3,0.0492,5.7109,7.3488,1.2868,9,1.5759
6,Σ,-,116,1.0000,116.0000,-,6.3908,-,122.3908


χ²_набл = 6.3908
χ²_набл (формула) = 6.3908


### 4. Доказать, что $$\chi^2_{набл} = \sum_{i=1}^{k} \frac{n_i^2}{n_i'} - n$$ Проконтролировать корректность вычисления $\chi^2_{набл}$.

Покажем, что формула:
$$\chi^2_{\text{набл}} = \sum_{i=1}^k \frac{(n_i - n_i')^2}{n_i'}$$
эквивалентна формуле:
$$\chi^2_{\text{набл}} = \sum_{i=1}^k \frac{n_i^2}{n_i'} - n$$

**Доказательство:**
Возведем в квадрат разность в числителе исходной суммы:
$$\chi^2_{\text{набл}} = \sum_{i=1}^k \frac{n_i^2 - 2n_i n_i' + (n_i')^2}{n_i'}$$
Разобьем дробь на три отдельных слагаемых:
$$\chi^2_{\text{набл}} = \sum_{i=1}^k \frac{n_i^2}{n_i'} - \sum_{i=1}^k \frac{2n_i n_i'}{n_i'} + \sum_{i=1}^k \frac{(n_i')^2}{n_i'}$$
Во втором и третьем слагаемом $n_i'$ сокращается:
$$\chi^2_{\text{набл}} = \sum_{i=1}^k \frac{n_i^2}{n_i'} - 2\sum_{i=1}^k n_i + \sum_{i=1}^k n_i'$$
Имеем $\sum_{i=1}^k n_i = n$ и $\sum_{i=1}^k n_i' = n$:
$$\chi^2_{\text{набл}} = \sum_{i=1}^k \frac{n_i^2}{n_i'} - 2n + n = \sum_{i=1}^k \frac{n_i^2}{n_i'} - n\quad\quad\quad\quad\blacksquare$$

In [9]:
def verify_chi2_calculation(chi2_main, chi2_alt, feature_name):
    print(f"Контроль вычислений для признака '{feature_name}':")
    print(f"χ²_набл (основная формула) = {chi2_main:.6f}")
    print(f"χ²_набл (альтернативная формула) = {chi2_alt:.6f}")
    
    diff = abs(chi2_main - chi2_alt)
    print(f"Разница = {diff:.10f}")
    
    if diff < 1e-5:
        print("Вычисления корректны")
    else:
        print("Расхождение")

In [10]:
verify_chi2_calculation(chi2_obs_nu, alt_chi2_nu, 'nu')

Контроль вычислений для признака 'nu':
χ²_набл (основная формула) = 0.729380
χ²_набл (альтернативная формула) = 0.729380
Разница = 0.0000000000
Вычисления корректны


In [11]:
verify_chi2_calculation(chi2_obs_E, alt_chi2_E, 'E')

Контроль вычислений для признака 'E':
χ²_набл (основная формула) = 6.390782
χ²_набл (альтернативная формула) = 6.390782
Разница = 0.0000000000
Вычисления корректны


### 5. По заданному уровню значимости $\alpha = 0.05$ и числу степеней свободы $df$ найти критическую точку $\chi^2_{крит}$ и сравнить с наблюдаемым значением. Сделать выводы.

In [12]:
def make_decision(chi2_obs, k_bins, feature_name, alpha=0.05):
    df_freedom = k_bins - 3
    chi2_crit = stats.chi2.ppf(1 - alpha, df_freedom)
    
    print(f"--- Признак: {feature_name} ---")
    print(f"Степени свободы = {df_freedom}")
    print(f"χ²_набл = {chi2_obs:.4f}")
    print(f"χ²_крит = {chi2_crit:.4f}")

In [13]:
make_decision(chi2_obs_nu, k_nu, 'nu')

--- Признак: nu ---
Степени свободы = 3
χ²_набл = 0.7294
χ²_крит = 7.8147


In [14]:
make_decision(chi2_obs_E, k_E, 'E')

--- Признак: E ---
Степени свободы = 3
χ²_набл = 6.3908
χ²_крит = 7.8147


**Выводы для обоих признаков**: 

$\chi^2_{набл}$ < $\chi^2_{крит}$. Нет оснований отвергать нулевую гипотезу (H0). Распределение можно считать нормальным.

## Вывод
В ходе выполнения практической работы были успешно рассчитаны доверительные интервалы для параметров распределения (математического ожидания и среднеквадратичного отклонения) исследуемых признаков `nu` и `E`. Подтверждено теоретическое правило: с увеличением уровня надежности $\gamma$ увеличивается ширина доверительного интервала.

Кроме того, была проведена статистическая проверка гипотезы о нормальности распределения с помощью критерия согласия Пирсона. Были рассчитаны теоретические вероятности попадания случайной величины в сгруппированные интервалы, построена расчетная таблица и вычислено наблюдаемое значение критерия. Сравнение наблюдаемого значения с критическим для $\alpha=0.05$ позволило сделать статистически обоснованные выводы о характере распределения исследуемых данных. Математически доказана и практически проверена эквивалентность двух формул вычисления критерия $\chi^2$.